# Keeping Every Measurement Next to Its Metadata
### 2.3 — pandas DataFrames for Sample Tables

A bare number — 0.83 AU — is almost useless. *Which* sample? *Which* batch? *Which*
operator, what concentration? In the lab a measurement always travels with its
metadata, and the fastest way to lose a result is to separate the two.

A pandas **DataFrame** is a table that keeps the number and its context in one row,
so they can't drift apart — and then lets you filter and summarize the way you'd
query a worklist.

> **One idea to hold onto:** keep the measurement and its metadata in one row of a
> DataFrame. Once they're together, "show me batch B" or "mean per operator" is a
> one-liner.

**By the end of this notebook you will be able to:**

1. Build a sample table and inspect it.
2. Filter rows by a condition (a batch, a concentration cut-off).
3. Summarize by group — per-batch mean, SD, and count.

## 1. Build a sample table

A small worklist: each row is one sample with its metadata and its measured
absorbance. We build it from a dictionary of columns.

In [ ]:
import numpy as np
import pandas as pd

samples = pd.DataFrame({
    "sample_id":      ["S-01", "S-02", "S-03", "S-04", "S-05", "S-06", "S-07", "S-08"],
    "batch":          ["A", "A", "A", "B", "B", "B", "C", "C"],
    "operator":       ["Lee", "Lee", "Patel", "Patel", "Lee", "Patel", "Lee", "Patel"],
    "concentration_mgL": [4.8, 5.1, 5.0, 7.9, 8.2, 8.0, 2.1, 2.0],
    "absorbance_AU":  [0.299, 0.318, 0.312, 0.490, 0.508, 0.497, 0.131, 0.124],
})

print("shape:", samples.shape, "(rows = samples, columns = fields)")
samples

## 2. Inspect and select

`.dtypes` shows each column's type; selecting a column gives a Series; `.loc` and
`.iloc` select rows by label and by position.

In [ ]:
print(samples.dtypes, "\n")

# One column (a Series), and a couple of selections.
print("mean absorbance:", round(samples["absorbance_AU"].mean(), 3), "AU")
print("\nfirst two rows:")
print(samples.iloc[:2][["sample_id", "concentration_mgL"]])

## 3. Filter rows by a condition

A condition like `samples["batch"] == "B"` produces a boolean mask; indexing the
DataFrame with it keeps the matching rows — the same masking idea as 2.2, now on a
table.

In [ ]:
# All of batch B.
batch_b = samples[samples["batch"] == "B"]
print("batch B has", len(batch_b), "samples")

# Combine conditions: concentration above 5 mg/L AND run by Lee.
# (Each condition in parentheses; & means 'and'.)
subset = samples[(samples["concentration_mgL"] > 5.0) & (samples["operator"] == "Lee")]
print("\nconc > 5 mg/L and operator Lee:")
print(subset[["sample_id", "batch", "concentration_mgL"]])

## 4. Summarize by group

`groupby` splits the table by a key, then you summarize each group. This is how you
turn a worklist into a per-batch report.

In [ ]:
# Per-batch concentration summary: mean, SD, and how many samples.
by_batch = samples.groupby("batch")["concentration_mgL"].agg(["mean", "std", "count"])
print("per-batch concentration (mg/L):")
print(by_batch.round(3))

# How many samples each operator ran.
print("\nsamples per operator:")
print(samples["operator"].value_counts())

## 5. Add a computed column

Results usually need a verdict. We add a pass/review flag against a spec window,
keeping it in the same row as the measurement it judges.

In [ ]:
spec_low, spec_high = 3.0, 9.0   # acceptance window, mg/L

# np.where(condition, value_if_true, value_if_false) -- a vectorized if/else.
samples["flag"] = np.where(
    (samples["concentration_mgL"] < spec_low) | (samples["concentration_mgL"] > spec_high),
    "review", "pass",
)
samples[["sample_id", "batch", "concentration_mgL", "flag"]]

## Key Takeaways

- A **DataFrame** keeps each measurement in one row with its metadata.
- Boolean conditions **filter** rows; combine them with `&` / `|` in parentheses.
- `groupby(...).agg([...])` turns a worklist into per-group summaries.
- A computed column (e.g. a pass/review flag) keeps the verdict beside the result.

## Practical Checklist

- [ ] Store sample ID, batch, operator, concentration, and signal in one table.
- [ ] Check `.dtypes` so numeric columns aren't accidentally text.
- [ ] Filter with parenthesized conditions; summarize with `groupby`.
- [ ] Keep derived columns (flags, ratios) in the same row as their inputs.

## Common Mistakes

- Splitting measurements from metadata into parallel lists that drift out of sync.
- Forgetting parentheses around each filter condition.
- Treating a numeric column that loaded as text — check `.dtypes` (see 2.4).

## Next Lesson

**2.4 — Reading Real Instrument Files (CSV, TXT, and the Messy Ones).** You can hold
a sample table; next we load one reliably from the files an instrument actually exports.